In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 5 - Week 8 Bayesian Optimisation
# --------------------------------------------------
#
# Function 5 has large-magnitude outputs, so Y is
# standardised manually before fitting the GP.
#
# The ordering of the objective is unchanged.
#
# Strategy:
# - maximise raw objective
# - standardise Y for numerical stability
# - ARD Matern GP
# - automatic hyperparameter optimisation
# - local + wide + global candidate generation
# - EI primary, UCB and GP mean diagnostic

In [2]:
X = np.load("function5/initial_inputs.npy")
Y = np.load("function5/initial_outputs.npy").reshape(-1)

assert len(X) == len(Y)
assert X.shape[1] == 4

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best observed input:")
print(best_x)

print("\nCurrent best observed output:")
print(best_y)

print("\nY range:")
print("min =", np.min(Y))
print("max =", np.max(Y))
print("std =", np.std(Y))

X shape: (27, 4)
Y shape: (27,)

Current best observed input:
[1. 1. 1. 1.]

Current best observed output:
8662.4825

Y range:
min = 0.1129397953712203
max = 8662.4825
std = 2442.12571125052


In [3]:
y_mean = np.mean(Y)
y_std = np.std(Y)

Y_scaled = (Y - y_mean) / y_std

best_y_scaled = np.max(Y_scaled)

print("Y mean:", y_mean)
print("Y std:", y_std)

print("\nBest scaled output:")
print(best_y_scaled)

Y mean: 1419.7303262966973
Y std: 2442.12571125052

Best scaled output:
2.965757307388801


In [4]:
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y_scaled)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
2.35**2 * Matern(length_scale=[1.66, 1.55, 1.92, 2], nu=2.5) + WhiteKernel(noise_level=0.0128)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


ARD lengthscales:
[1.65931569 1.55194255 1.9207481  2.        ]

Normalised inverse-lengthscale sensitivity:
[0.26576417 0.28415141 0.22959109 0.22049333]


In [6]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)


Local widths: [0.1 0.1 0.1 0.1]
Wide widths: [0.2 0.2 0.2 0.2]


In [7]:
rng = np.random.default_rng(42)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(70000, 4)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(40000, 4)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(70000, 4)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

# Explicitly include all corners because the current
# best itself is a corner.
corners = np.array([
    [a, b, c, d]
    for a in [0.0, 1.0]
    for b in [0.0, 1.0]
    for c in [0.0, 1.0]
    for d in [0.0, 1.0]
])

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates,
    corners
])

print(
    "Candidates before filtering:",
    len(candidates)
)

Candidates before filtering: 180016


In [8]:
tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 169998


In [9]:
mu_scaled, sigma_scaled = gp.predict(
    candidates,
    return_std=True
)

# Convert predictions back to raw objective scale
mu_raw = (
    mu_scaled * y_std
    + y_mean
)

sigma_raw = (
    sigma_scaled * y_std
)

print("Predictions complete.")

Predictions complete.


In [10]:
def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI

In [11]:
EI = expected_improvement(
    mu_scaled,
    sigma_scaled,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\nPRIMARY EI RESULT")

print("candidate =", candidates[ei_idx])
print("predicted raw mean =", mu_raw[ei_idx])
print("predicted raw std =", sigma_raw[ei_idx])
print("EI scaled =", EI[ei_idx])


PRIMARY EI RESULT
candidate = [1.        1.        1.        0.9899924]
predicted raw mean = 8237.87021279142
predicted raw std = 343.88592351500284
EI scaled = 0.0073532700878419475


In [12]:
xi_values = [
    0.0,
    0.01,
    0.05,
    0.10
]

print("\nEI sensitivity check:\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu_scaled,
        sigma_scaled,
        best_y_scaled,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", xi,
        "\n candidate =", candidates[idx],
        "\n raw mean =", round(mu_raw[idx], 3),
        "\n raw std =", round(sigma_raw[idx], 3),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI sensitivity check:

xi = 0.0 
 candidate = [1.        1.        1.        0.9899924] 
 raw mean = 8237.87 
 raw std = 343.886 
 EI = 0.00735327 

xi = 0.01 
 candidate = [1.        1.        1.        0.9899924] 
 raw mean = 8237.87 
 raw std = 343.886 
 EI = 0.00633283 

xi = 0.05 
 candidate = [1.        1.        1.        0.9899924] 
 raw mean = 8237.87 
 raw std = 343.886 
 EI = 0.00335212 

xi = 0.1 
 candidate = [1.        1.        1.        0.9899924] 
 raw mean = 8237.87 
 raw std = 343.886 
 EI = 0.00138409 



In [13]:
print("\nUCB diagnostic:\n")

for beta in [
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n raw mean =", round(mu_raw[idx], 3),
        "\n raw std =", round(sigma_raw[idx], 3),
        "\n scaled UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB diagnostic:

beta=0.1 
 candidate = [1.        1.        1.        0.9899924] 
 raw mean = 8237.87 
 raw std = 343.886 
 scaled UCB = 2.805969 

beta=0.25 
 candidate = [1.        1.        1.        0.9899924] 
 raw mean = 8237.87 
 raw std = 343.886 
 scaled UCB = 2.827091 

beta=0.5 
 candidate = [1.        1.        1.        0.9899924] 
 raw mean = 8237.87 
 raw std = 343.886 
 scaled UCB = 2.862294 

beta=1.0 
 candidate = [1.        1.        1.        0.9899924] 
 raw mean = 8237.87 
 raw std = 343.886 
 scaled UCB = 2.932702 



In [14]:
mean_idx = np.argmax(mu_scaled)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("raw mean =", mu_raw[mean_idx])
print("raw std =", sigma_raw[mean_idx])


Highest predicted mean:
candidate = [1.        1.        1.        0.9899924]
raw mean = 8237.87021279142
raw std = 343.88592351500284


In [15]:
# --------------------------------------------------
# Final Function 5 Week 8 selection
# --------------------------------------------------
#
# EI, all tested UCB settings and the highest GP mean
# select the same candidate.
#
# The current best [1,1,1,1] is already observed, so
# the near-duplicate filter forces the acquisition to
# the nearest admissible point.
#
# Strong agreement across acquisition criteria means
# no additional trust-region correction is needed.

final_idx = np.argmax(EI)

week8_candidate = candidates[final_idx]

print("Week 8 Function 5 candidate:")
print(week8_candidate)

print("\nPredicted raw mean:")
print(mu_raw[final_idx])

print("\nPredicted raw std:")
print(sigma_raw[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week8_candidate
)

print("\nPortal format:")
print(portal)

Week 8 Function 5 candidate:
[1.        1.        1.        0.9899924]

Predicted raw mean:
8237.87021279142

Predicted raw std:
343.88592351500284

Portal format:
1.000000-1.000000-1.000000-0.989992
